In [1]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.prompts import ChatPromptTemplate

from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_core.output_parsers import StrOutputParser

from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_classic.storage import InMemoryStore

In [3]:
import os

In [4]:
questions = [
    "Qual é a visão de Euclides da Cunha sobre o ambiente natural do sertão nordestino e como ele influencia a vida dos habitantes?",
    "Quais são as principais características da população sertaneja descritas por Euclides da Cunha? Como ele relaciona essas características com o ambiente em que vivem?",
    "Qual foi o contexto histórico e político que levou à Guerra de Canudos, segundo Euclides da Cunha?",
    "Como Euclides da Cunha descreve a figura de Antônio Conselheiro e seu papel na Guerra de Canudos?",
    "Quais são os principais aspectos da crítica social e política presentes em \"Os Sertões\"? Como esses aspectos refletem a visão do autor sobre o Brasil da época?",
]

In [ ]:
## OpenAI Key
os.environ["OPENAI_API_KEY"] = ""

In [6]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
llm = ChatOpenAI(model="gpt-3.5-turbo", max_tokens=512)

In [7]:
pdf_link = '../os-sertoes.pdf'
loader = PyPDFLoader(pdf_link)
pages = loader.load_and_split()

In [9]:
# Chunking

child_splitter = RecursiveCharacterTextSplitter(chunk_size=200)
parent_splitter = RecursiveCharacterTextSplitter(chunk_size=4000, chunk_overlap=200)
lenght_function = len
add_start_index = True

In [11]:
# Storage

store = InMemoryStore()
vectorstore = Chroma(embedding_function=embeddings, persist_directory='vectorDB')

In [15]:
child_splitter = child_splitter[0] if isinstance(child_splitter, tuple) else child_splitter
parent_splitter = parent_splitter[0] if isinstance(parent_splitter, tuple) else parent_splitter

parent_document_retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    docstore=store,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
)

# Avoid Chroma max batch-size error by indexing in smaller batches
batch_size = 20  # reduce if needed

for i in range(0, len(pages), batch_size):
    parent_document_retriever.add_documents(pages[i:i + batch_size], ids=None)

In [17]:
# Prompt
TEMPLATE = """
    Você é um assistente de perguntas e respostas sobre o livro "Os Sertões" de Euclides da Cunha. Responda a pergunta abaixo utilizando o contexto informado.
    Context: {context}
    Pergunta: {question}
"""

prompt = ChatPromptTemplate.from_template(TEMPLATE)

In [19]:
setup_retrieval = RunnableParallel({"question": RunnablePassthrough(), "context": parent_document_retriever})
output_parser = StrOutputParser()

In [20]:
parent_chain_retrieval = setup_retrieval | prompt | llm | output_parser

In [24]:
for index, question in enumerate(questions):
    result = {"output": parent_chain_retrieval.invoke(question)}
    answer = result["output"]
    print({"numero": index+1, "pergunta": question, "resposta": answer})

{'numero': 1, 'pergunta': 'Qual é a visão de Euclides da Cunha sobre o ambiente natural do sertão nordestino e como ele influencia a vida dos habitantes?', 'resposta': 'Euclides da Cunha descreve o ambiente natural do sertão nordestino como uma região árida e desafiadora, marcada por secas, incêndios e uma vegetação escassa. Ele destaca a influência dessas condições adversas na vida dos habitantes locais, que enfrentam grandes dificuldades para sobreviver nesse ambiente hostil. A descrição feita por Euclides da Cunha destaca a dureza da vida no sertão e a luta constante dos habitantes para enfrentar as adversidades naturais.'}
{'numero': 2, 'pergunta': 'Quais são as principais características da população sertaneja descritas por Euclides da Cunha? Como ele relaciona essas características com o ambiente em que vivem?', 'resposta': 'Euclides da Cunha descreve a população sertaneja como fortes, destacando que eles não possuem o raquitismo exaustivo dos mestiços neurastênicos do litoral. A